In [15]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.metrics import mean_absolute_error, mean_squared_error


ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from dahlia.imputation.baselines import (
    MeanImputer,
    ClusterMeanImputer,
)

from dahlia.imputation.knn import (
    KNNImputer,
    ClusterKNNImputer,
)

from dahlia.imputation.iterative import (
    RandomForestImputer,
    MICEImputer,
)

In [16]:
iris = load_iris(as_frame=True)

X_original = iris.data.rename(
    columns={
        "sepal length (cm)": "SepalLengthCm",
        "sepal width (cm)": "SepalWidthCm",
        "petal length (cm)": "PetalLengthCm",
        "petal width (cm)": "PetalWidthCm",
    }
)

TARGET_COLUMN = "SepalLengthCm"

REFERENCE_COLUMNS = [
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm",
]

X_original.head()

,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [ ]:
SET_01 = [
    114, 62, 33, 107, 7,
    100, 40, 86, 76, 71,
    134, 51, 73, 54, 63,
]

SET_02 = [
    114, 62, 33, 107, 7,
    100, 40, 86, 76, 71,
    134, 51, 73, 54, 63,
    37, 78, 90, 45, 16,
    121, 66, 24, 8, 126,
    22, 44, 97, 93, 26,
]

SET_03 = [
    114, 62, 33, 107, 7,
    100, 40, 86, 76, 71,
    134, 51, 73, 54, 63,
    37, 78, 90, 45, 16,
    121, 66, 24, 8, 126,
    22, 44, 97, 93, 26,
    137, 84, 27, 127, 132,
    59, 18, 83, 61, 92,
    112, 2, 141, 43, 10,
]

SETS = {
    "NA 0.1": SET_01,
    "NA 0.2": SET_02,
    "NA 0.3": SET_03,
}

#Should be 15, 30, 45
print(len(SET_01),len(SET_02),len(SET_03))


15 30 45


In [18]:
X_missing_sets = {}

for set_name, deleted_rows in SETS.items():
    X_missing = X_original.copy()

    X_missing.loc[
        deleted_rows,
        TARGET_COLUMN,
    ] = np.nan

    X_missing_sets[set_name] = X_missing


for set_name, X_missing in X_missing_sets.items():
    print(set_name, X_missing[TARGET_COLUMN].isna().sum())

NA 0.1 15
NA 0.2 30
NA 0.3 45


# Test Configuration

In [19]:
SEED = 0
NUMBER_OF_RUNS = 10

N_NEIGHBORS = 5
N_CLUSTERS = 3

METHOD_NAMES = [
    "Mean",
    "Cluster Mean",
    "KNN",
    "Cluster KNN",
    "Random Forest",
    "MICE",
]

In [ ]:
results = []
predictions = {}

for set_name, deleted_rows in SETS.items():
    X_missing = X_missing_sets[set_name]

    true_values = X_original.loc[
        deleted_rows,
        TARGET_COLUMN,
    ].to_numpy()

    predictions[set_name] = {
    method_name: []
    for method_name in METHOD_NAMES
    }

    for run in range(1, NUMBER_OF_RUNS + 1):
        np.random.seed(SEED)

        imputers = {
            "Mean": MeanImputer(),
            "Cluster Mean": ClusterMeanImputer(
                n_clusters=N_CLUSTERS,
                random_state=SEED,
            ),
            "KNN": KNNImputer(
                n_neighbors=N_NEIGHBORS,
            ),
            "Cluster KNN": ClusterKNNImputer(
                n_clusters=N_CLUSTERS,
                n_neighbors=N_NEIGHBORS,
                random_state=SEED,
            ),
            "Random Forest": RandomForestImputer(
                n_estimators=100,
                max_depth=None,
                max_iter=10,
                random_state=SEED,
            ),
            "MICE": MICEImputer(
                max_iter=10,
                tol=1e-3,
                sample_posterior=False,
                random_state=SEED,
            ),
        }

        for method_name, imputer in imputers.items():
            X_imputed = imputer.fit_transform(X_missing)

            pd.testing.assert_frame_equal(
                X_imputed[REFERENCE_COLUMNS],
                X_original[REFERENCE_COLUMNS],
            )

            predicted_values = X_imputed.loc[
                deleted_rows,
                TARGET_COLUMN,
            ].to_numpy()

            predictions[set_name][method_name].append(
                predicted_values.copy()
            )

            mae = mean_absolute_error(
                true_values,
                predicted_values,
            )

            rmse = np.sqrt(
                mean_squared_error(
                    true_values,
                    predicted_values,
                )
            )

            results.append(
                {
                    "set": set_name,
                    "method": method_name,
                    "run": run,
                    "seed": SEED,
                    "deleted_values": len(deleted_rows),
                    "mae": mae,
                    "rmse": rmse,
                }
            )

results_df = pd.DataFrame(results)

results_df

c:\Users\Stefan\Desktop\DAHLIA-2026\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Stefan\Desktop\DAHLIA-2026\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Stefan\Desktop\DAHLIA-2026\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
c:\Users\Stefan\Desktop\DAHLIA-2026\venv\Lib\site-pa

,set,method,run,seed,deleted_values,mae,rmse
0,NA 0.1,Mean,1,0,15,0.560444,0.673663
1,NA 0.1,Cluster Mean,1,0,15,0.503016,0.598472
2,NA 0.1,KNN,1,0,15,0.365333,0.441875
3,NA 0.1,Cluster KNN,1,0,15,0.425333,0.516165
4,NA 0.1,Random Forest,1,0,15,0.325847,0.379846
...,...,...,...,...,...,...,...
175,NA 0.3,Cluster Mean,10,0,45,0.361659,0.454621
176,NA 0.3,KNN,10,0,45,0.288000,0.356052
177,NA 0.3,Cluster KNN,10,0,45,0.309333,0.377548
178,NA 0.3,Random Forest,10,0,45,0.285767,0.360731


In [21]:
reproducibility = []

for set_name in SETS:
    for method_name in METHOD_NAMES:
        reference = predictions[set_name][method_name][0]

        all_runs_equal = True
        maximum_difference = 0.0

        for current_result in predictions[set_name][method_name][1:]:
            current_equal = np.allclose(
                reference,
                current_result,
                rtol=0.0,
                atol=1e-12,
            )

            all_runs_equal = (
                all_runs_equal
                and current_equal
            )

            difference = np.max(
                np.abs(
                    reference - current_result
                )
            )

            maximum_difference = max(
                maximum_difference,
                float(difference),
            )

        reproducibility.append(
            {
                "set": set_name,
                "method": method_name,
                "all_10_runs_equal": all_runs_equal,
                "maximum_difference": maximum_difference,
            }
        )

reproducibility_df = pd.DataFrame(reproducibility)

reproducibility_df

,set,method,all_10_runs_equal,maximum_difference
0,NA 0.1,Mean,True,0.000000e+00
1,NA 0.1,Cluster Mean,True,0.000000e+00
2,NA 0.1,KNN,True,0.000000e+00
3,NA 0.1,Cluster KNN,True,0.000000e+00
4,NA 0.1,Random Forest,True,1.776357e-15
5,NA 0.1,MICE,True,0.000000e+00
6,NA 0.2,Mean,True,0.000000e+00
7,NA 0.2,Cluster Mean,True,0.000000e+00
8,NA 0.2,KNN,True,0.000000e+00
9,NA 0.2,Cluster KNN,True,0.000000e+00


In [22]:
assert reproducibility_df["all_10_runs_equal"].all()
assert (
    reproducibility_df["maximum_difference"] <= 1e-12
).all()

print("Wszystkie metody dały te same wyniki w 10 runach.")

Wszystkie metody dały te same wyniki w 10 runach.


In [23]:
first_run_results = (
    results_df[results_df["run"] == 1]
    .sort_values(["set", "rmse"])
    .reset_index(drop=True)
)

first_run_results[
    [
        "set",
        "method",
        "deleted_values",
        "mae",
        "rmse",
    ]
]

,set,method,deleted_values,mae,rmse
0,NA 0.1,MICE,15,0.279994,0.339475
1,NA 0.1,Random Forest,15,0.325847,0.379846
2,NA 0.1,KNN,15,0.365333,0.441875
3,NA 0.1,Cluster KNN,15,0.425333,0.516165
4,NA 0.1,Cluster Mean,15,0.503016,0.598472
5,NA 0.1,Mean,15,0.560444,0.673663
6,NA 0.2,MICE,30,0.271937,0.328141
7,NA 0.2,KNN,30,0.283333,0.354420
8,NA 0.2,Random Forest,30,0.311928,0.376304
9,NA 0.2,Cluster KNN,30,0.314667,0.389838


In [31]:
summary = (
    results_df
    .groupby(
        ["set", "method"],
        as_index=False,
    )
    .agg(
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
    )
    
)

summary

,set,method,mean_mae,std_mae,mean_rmse,std_rmse
0,NA 0.1,Cluster KNN,0.425333,0.000000e+00,0.516165,0.000000e+00
1,NA 0.1,Cluster Mean,0.503016,0.000000e+00,0.598472,0.000000e+00
2,NA 0.1,KNN,0.365333,0.000000e+00,0.441875,0.000000e+00
3,NA 0.1,MICE,0.279994,0.000000e+00,0.339475,0.000000e+00
4,NA 0.1,Mean,0.560444,0.000000e+00,0.673663,0.000000e+00
5,NA 0.1,Random Forest,0.325847,1.199178e-16,0.379846,1.046728e-16
6,NA 0.2,Cluster KNN,0.314667,0.000000e+00,0.389838,0.000000e+00
7,NA 0.2,Cluster Mean,0.401863,0.000000e+00,0.492163,0.000000e+00
8,NA 0.2,KNN,0.283333,0.000000e+00,0.354420,0.000000e+00
9,NA 0.2,MICE,0.271937,0.000000e+00,0.328141,0.000000e+00


Przy poprawnie wykonanych testach std_mae~0 oraz std_